<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/libros/ciencia-de-datos-con-python/volumen-02/cuadernos/capitulo-031-rankings-valores-extremos-y-outliers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 31 — Rankings, valores extremos y outliers

## Ordenar para detectar valores destacados

En los capítulos anteriores usamos tablas agrupadas para comparar categorías.
Vimos que una misma categoría puede destacarse según distintos indicadores: cantidad de registros, total facturado, promedio de cuenta o promedio de propina.

Ahora vamos a trabajar con una idea relacionada: construir rankings. Un ranking es una lista ordenada según un criterio. Ese criterio puede ser, por ejemplo:

- mayor facturación total;
- mayor promedio de cuenta;
- mayor propina;
- menor cantidad de registros;
- cuentas más altas o más bajas.

Ordenar los datos nos ayuda a detectar rápidamente valores destacados. Pero también exige interpretar con cuidado: un valor alto, bajo o inusual no siempre significa lo mismo.

## Cargar el dataset de trabajo

Vamos a seguir usando el dataset `tips` de Seaborn.

Como ya conocemos sus variables principales, podemos concentrarnos en ordenar resultados, construir rankings y observar valores extremos.

In [1]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("tips")

df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


La tabla muestra las primeras filas del dataset.

Seguimos trabajando con las mismas variables de los capítulos anteriores. En este capítulo vamos a ordenar datos para detectar valores destacados. Algunas veces vamos a ordenar tablas de resumen, donde cada fila representa una categoría o grupo. Otras veces vamos a ordenar registros individuales del dataset.

En ambos casos, lo importante será aclarar siempre cuál es el criterio de ordenamiento.

## Construir una tabla base para rankings por día

Para empezar, vamos a construir una tabla de resumen por día. Esta tabla será parecida a las que usamos en capítulos anteriores, pero ahora la vamos a usar como base para construir rankings. Calcularemos:

- cantidad de cuentas;
- total facturado;
- promedio de cuenta;
- promedio de propina.

In [2]:
resumen_dias = df.groupby(
    "day",
    observed=True
).agg(
    cantidad_cuentas=("total_bill", "count"),
    total_facturado=("total_bill", "sum"),
    promedio_cuenta=("total_bill", "mean"),
    promedio_propina=("tip", "mean")
)

resumen_dias = resumen_dias.round(2).reset_index()

resumen_dias

,day,cantidad_cuentas,total_facturado,promedio_cuenta,promedio_propina
0,Thur,62,1096.33,17.68,2.77
1,Fri,19,325.88,17.15,2.73
2,Sat,87,1778.40,20.44,2.99
3,Sun,76,1627.16,21.41,3.26


La tabla resume los principales indicadores por día. Cada fila representa un día y cada columna muestra un indicador distinto. Esta tabla será nuestra base para construir rankings. Según el criterio que elijamos, el orden de los días puede cambiar.

Por ejemplo, podemos ordenar por `total_facturado` si queremos saber qué día acumuló más dinero, o por `promedio_cuenta` si queremos saber qué día tuvo cuentas más altas en promedio.

## Ranking por facturación total

Un primer ranking posible es ordenar los días según el total facturado. La pregunta que queremos responder es:

**¿Qué día acumuló más facturación total?**

Para eso vamos a ordenar la tabla `resumen_dias` por la columna `total_facturado`, de mayor a menor.

In [3]:
ranking_facturacion = resumen_dias.sort_values(
    by="total_facturado",
    ascending=False
)

ranking_facturacion

,day,cantidad_cuentas,total_facturado,promedio_cuenta,promedio_propina
2,Sat,87,1778.40,20.44,2.99
3,Sun,76,1627.16,21.41,3.26
0,Thur,62,1096.33,17.68,2.77
1,Fri,19,325.88,17.15,2.73


El ranking por `total_facturado` muestra que:

- `Sat` ocupa el primer lugar, con **1778.40**.
- `Sun` ocupa el segundo lugar, con **1627.16**.
- `Thur` ocupa el tercer lugar, con **1096.33**.
- `Fri` queda en último lugar, con **325.88**.

Este ranking responde una pregunta específica: qué día acumuló más facturación total.

También conviene mirar la columna `cantidad_cuentas`. `Sat` no solo tiene la mayor facturación total, sino también la mayor cantidad de cuentas registradas. Esto ayuda a interpretar por qué aparece en primer lugar.


## Ranking por promedio de cuenta

Ahora vamos a ordenar los días según otro criterio: el promedio de cuenta. La pregunta cambia:

**¿Qué día tuvo el mayor valor promedio por cuenta?**

El ranking puede cambiar porque el promedio no mide lo mismo que el total facturado.

In [4]:
ranking_promedio_cuenta = resumen_dias.sort_values(
    by="promedio_cuenta",
    ascending=False
)

ranking_promedio_cuenta

,day,cantidad_cuentas,total_facturado,promedio_cuenta,promedio_propina
3,Sun,76,1627.16,21.41,3.26
2,Sat,87,1778.40,20.44,2.99
0,Thur,62,1096.33,17.68,2.77
1,Fri,19,325.88,17.15,2.73


El ranking por `promedio_cuenta` cambia el orden. Ahora el primer lugar corresponde a `Sun`, con un promedio de cuenta de **21.41**. Luego aparecen:

- `Sat`, con **20.44**;
- `Thur`, con **17.68**;
- `Fri`, con **17.15**.

Este ranking no responde qué día facturó más en total, sino qué día tuvo cuentas más altas en promedio. La diferencia es importante: `Sat` acumula más facturación total, pero `Sun` tiene el promedio de cuenta más alto.

## Seleccionar los primeros lugares de un ranking

A veces no necesitamos ver el ranking completo. Si solo queremos observar los primeros lugares, podemos ordenar la tabla y luego usar `head()`.

Por ejemplo, vamos a quedarnos con los **2 días con mayor facturación total**.

In [5]:
ranking_facturacion.head(2)

,day,cantidad_cuentas,total_facturado,promedio_cuenta,promedio_propina
2,Sat,87,1778.40,20.44,2.99
3,Sun,76,1627.16,21.41,3.26


La salida muestra los **2 primeros lugares** del ranking por facturación total. Los días con mayor facturación total son:

- `Sat`, con **1778.40**;
- `Sun`, con **1627.16**.

Esta selección es útil cuando queremos concentrarnos solo en las categorías más destacadas según un criterio. En este caso, el criterio es `total_facturado`. Si cambiamos el criterio de ordenamiento, también puede cambiar el ranking.

## Seleccionar los últimos lugares de un ranking

También podemos mirar el otro extremo del ranking. Si una tabla ya está ordenada de mayor a menor, `tail()` permite ver los últimos registros.

Ahora vamos a observar los **2 días con menor facturación total**.

In [6]:
ranking_facturacion.tail(2)

,day,cantidad_cuentas,total_facturado,promedio_cuenta,promedio_propina
0,Thur,62,1096.33,17.68,2.77
1,Fri,19,325.88,17.15,2.73


La salida muestra los **2 últimos lugares** del ranking por facturación total. En este caso aparecen:

- `Thur`, con **1096.33**;
- `Fri`, con **325.88**.

Esto permite observar el extremo inferior del ranking. Sin embargo, también conviene mirar la cantidad de cuentas. `Fri` tiene mucha menos facturación total, pero también tiene solo **19 cuentas** registradas. Por eso, el total facturado debe interpretarse junto con la cantidad de registros.

## Rankings de registros individuales

Hasta ahora ordenamos una tabla de resumen por día. También podemos ordenar el dataset original para observar registros individuales destacados. Por ejemplo, podemos preguntarnos:

**¿Cuáles fueron las cuentas más altas del dataset?**

Para responderlo vamos a ordenar el `DataFrame` original por `total_bill`, de mayor a menor.

In [7]:
cuentas_mas_altas = df.sort_values(
    by="total_bill",
    ascending=False
)

cuentas_mas_altas.head()

,total_bill,tip,sex,smoker,day,time,size
170,50.81,10.00,Male,Yes,Sat,Dinner,3
212,48.33,9.00,Male,No,Sat,Dinner,4
59,48.27,6.73,Male,No,Sat,Dinner,4
156,48.17,5.00,Male,No,Sun,Dinner,6
182,45.35,3.50,Male,Yes,Sun,Dinner,3


La tabla muestra las **5 cuentas más altas** del dataset. La cuenta más alta tiene un `total_bill` de **50.81** y una propina de **10.00**. Luego aparecen otras cuentas cercanas a **48** y **45**.

También podemos observar que estas cuentas altas aparecen en `Dinner`, principalmente durante `Sat` y `Sun`. Estos registros son valores destacados dentro del dataset. No sabemos todavía si son valores extremos en sentido estricto, pero sí sabemos que ocupan los primeros lugares cuando ordenamos por `total_bill`.

Este tipo de ranking puede servir para detectar casos que conviene mirar con más atención.


## Observar las cuentas más bajas

También podemos ordenar en el sentido contrario para observar las cuentas más bajas del dataset. Esto permite revisar el otro extremo de la distribución.

In [8]:
cuentas_mas_bajas = df.sort_values(
    by="total_bill",
    ascending=True
)

cuentas_mas_bajas.head()

,total_bill,tip,sex,smoker,day,time,size
67,3.07,1.00,Female,Yes,Sat,Dinner,1
92,5.75,1.00,Female,Yes,Fri,Dinner,2
111,7.25,1.00,Female,No,Sat,Dinner,1
172,7.25,5.15,Male,Yes,Sun,Dinner,2
149,7.51,2.00,Male,No,Thur,Lunch,2


La tabla muestra las **5 cuentas más bajas** del dataset. La cuenta más baja tiene un `total_bill` de **3.07**. Luego aparecen cuentas de **5.75**, **7.25**, **7.25** y **7.51**.

Estos registros se encuentran en distintos días y horarios, aunque todos corresponden a mesas pequeñas, de **1** o **2** personas. Como ocurría con las cuentas más altas, estos valores no deben interpretarse automáticamente como errores. Por ahora solo sabemos que son los valores más bajos cuando ordenamos la columna `total_bill`.

El ordenamiento nos ayuda a detectar casos destacados, pero todavía necesitamos contexto para decidir si son valores esperables, raros o problemáticos.

## Valores extremos y outliers

Cuando ordenamos los datos, los valores extremos aparecen con facilidad. Un valor extremo es un dato que está muy por encima o muy por debajo de la mayoría de los valores observados.

En análisis de datos también suele usarse el término **outlier** para referirse a valores que se apartan de manera marcada del comportamiento general del dataset. Pero detectar un valor extremo no significa que debamos eliminarlo.

Un outlier puede ser:

- un error de carga o medición;
- un caso real poco frecuente;
- una situación especial que merece análisis;
- un dato válido, pero muy diferente del resto.

Por eso, antes de modificar o eliminar un valor extremo, conviene observarlo con más atención.

## Observar un valor extremo con más contexto

Vamos a revisar el registro con la cuenta más alta del dataset. La idea no es modificarlo ni eliminarlo, sino observar qué información lo acompaña.

In [9]:
df.loc[df["total_bill"].idxmax()]

,170
total_bill,50.81
tip,10.0
sex,Male
smoker,Yes
day,Sat
time,Dinner
size,3


El registro con la cuenta más alta tiene un `total_bill` de **50.81**. Al observar el resto de las columnas, vemos que corresponde a una mesa de **3 personas**, durante `Dinner`, un sábado (`Sat`). También registra una propina de **10.00**.

Esta información ayuda a contextualizar el valor. No estamos mirando solo un número aislado, sino el registro completo. Por ahora no observamos ningún elemento que indique automáticamente que este dato sea un error.

## Observar el valor mínimo

También podemos revisar el registro con la cuenta más baja. Esto nos permite mirar el otro extremo de la columna `total_bill` y analizarlo con el mismo criterio: observar antes de decidir.

In [10]:
df.loc[df["total_bill"].idxmin()]

,67
total_bill,3.07
tip,1.0
sex,Female
smoker,Yes
day,Sat
time,Dinner
size,1


El registro con la cuenta más baja tiene un `total_bill` de **3.07**. Al observar el resto de las columnas, vemos que corresponde a una mesa de **1 persona**, durante `Dinner`, un sábado (`Sat`). La propina registrada es de **1.00**.

Este valor es bajo en comparación con el resto del dataset, pero el contexto ayuda a interpretarlo: una cuenta pequeña puede ser razonable si corresponde a una sola persona.

Nuevamente, no alcanza con ver que un valor está en un extremo para decidir que es un error. Primero debemos observarlo dentro del contexto de las demás variables.

## Qué hacer con un outlier

Detectar un valor extremo no significa modificarlo automáticamente. Antes de decidir qué hacer con un outlier, conviene hacerse algunas preguntas:

- ¿el valor es posible dentro del contexto?
- ¿hay otras columnas que ayuden a explicarlo?
- ¿podría ser un error de carga o medición?
- ¿afecta mucho los promedios u otros indicadores?
- ¿el objetivo del análisis requiere conservarlo, marcarlo o tratarlo de alguna manera?

En algunos casos, un outlier puede ser un error y habrá que corregirlo o excluirlo. En otros casos, puede ser un dato real y valioso para el análisis. Por eso, en esta etapa no vamos a eliminar estos valores. Los vamos a interpretar como señales para observar con más atención.

## Cierre del capítulo

En este capítulo trabajamos con rankings, valores extremos y outliers. Vimos que un ranking es una lista ordenada según un criterio. Ese criterio debe estar claro: facturación total, promedio de cuenta, propina, cantidad de registros u otra variable.

También vimos que podemos construir rankings de categorías, a partir de tablas agrupadas, o rankings de registros individuales, ordenando directamente el dataset original. Ordenar los datos nos ayuda a detectar valores destacados: los más altos, los más bajos o los que se apartan del comportamiento general.

A esos valores que se alejan mucho del resto solemos llamarlos **valores extremos** u **outliers**. Pero un outlier no es automáticamente un error. Puede ser un dato incorrecto, un caso raro, una situación especial o simplemente un valor válido pero poco frecuente. Por eso, antes de eliminar o modificar un valor extremo, conviene observarlo, contextualizarlo y decidir según el objetivo del análisis.

En los próximos capítulos vamos a seguir construyendo herramientas para resumir, comparar y comunicar resultados de manera más clara.